# Likelihood Profiling of Exponential Mixtures

This supporting notebook maps the likelihood surface of finite exponential mixtures in the high-mass diphoton example. It fixes selected raw rate parameters on a grid, profiles the remaining free parameters, and visualizes the resulting negative-log-likelihood projections.

## Roadmap

1. Load the unbinned diphoton dataset and compute its mean for mixture initialization.
2. Define two- and three-component parameter grids in the raw-rate parameterization.
3. Run `scan_parameters`, which writes a temporary RooWorkspace and distributes grid points through the task helper.
4. Plot pairwise profile projections to inspect degeneracies and local structure.

Profile scans can be computationally intensive; use a smaller grid or fewer batches for exploratory runs.

In [1]:
# Imports
import ROOT
import array

from tools import scale_out as so

from emm import data
from emm import fitting
from emm import models

import matplotlib.pyplot as plt
import numpy as np

import numpy as np
import matplotlib.pyplot as plt

Welcome to JupyROOT 6.30/04


In [ ]:
# Load data
x = ROOT.RooRealVar("x", "Diphoton Mass [GeV]", 500, 10_000)
data_tree = data.get_diphoton_data(tree=True)
diphoton_data = ROOT.RooDataSet("mgg", "mgg", ROOT.RooArgSet(x), ROOT.RooFit.Import(data_tree))
n = diphoton_data.numEntries()
data_mean = diphoton_data.mean(x)
print(f"Data mean: {data_mean}")

## Two-Component Scan

The first grid fixes the two raw rate coordinates in turn and profiles the remaining fit parameters at each grid point. The raw-rate representation is used by the model implementation; the profile plot therefore exposes the optimizer geometry in its native coordinates rather than in transformed physical rates.

In [ ]:
pset = {
    'raw_rate_0': (0.3, 3, 50),
    'raw_rate_1': (0.3, 3, 50),
}

# def model_primitive(x, data):
#     model = emm.ExponentialMixtureModel(x, 2, data_mean=data.mean(x),)
#     return model
model = emm.ExponentialMixtureModel(x, 2, data_mean=data.mean(x),)
df = emm.scan_parameters(model, data, pset)
fig, axes = emm.plot_pair_profiles(df, pset)

## Three-Component Extension

This extension adds a third raw rate and repeats the same profiling procedure. The denser parameter space is useful for identifying shallow directions and competing minima that motivate randomized restarts in the main fitting workflows.

In [ ]:

pset = {
    'raw_rate_0': (0.3, 3, 20),
    'raw_rate_1': (0.3, 3, 20),
    'raw_rate_2': (0.3, 3, 20),
}

model = emm.ExponentialMixtureModel(x, 3, data_mean=data.mean(x),)
df = emm.scan_parameters(model, data, pset)
fig, axes = emm.plot_pair_profiles(df, pset)